In [29]:
from huggingface_hub import login
login()

In [31]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="Dipak654/gpt-8-layer",
    filename="gpt-8-layer-weights.pt"
)

print(weights_path)

gpt-8-layer-weights.pt:   0%|          | 0.00/308M [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--Dipak654--gpt-8-layer/snapshots/b3cc424a8d9855651d0a40d3a82a503e55788d38/gpt-8-layer-weights.pt


In [45]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [32]:
# gpt2_min.py
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import pdb

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.resid_drop = nn.Dropout(config.dropout)
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        # pdb.set_trace()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.size()
        # pdb.set_trace()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.wte(idx) + self.wpe(pos)
        x = self.drop(x)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        # pdb.set_trace()
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        # pdb.set_trace()
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            # pdb.set_trace()
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                thresh = v[:, -1].unsqueeze(-1)
                logits = torch.where(logits < thresh, torch.full_like(logits, -float("inf")), logits)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            # pdb.set_trace()
            idx = torch.cat((idx, next_token), dim=1)
        return idx

In [33]:
config = GPTConfig(
    vocab_size=50257,
    block_size=128,
    n_layer=8,
    n_head=8,
    n_embd=512
)

model = GPT(config)

In [34]:
state_dict = torch.load(
    weights_path,
    map_location=device
)

In [36]:
model.load_state_dict(state_dict)

<All keys matched successfully>

In [37]:
model.to(device)

GPT(
  (wte): Embedding(50257, 512)
  (wpe): Embedding(128, 512)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-7): 8 x Block(
      (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=512, out_features=1536, bias=True)
        (c_proj): Linear(in_features=512, out_features=512, bias=True)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc): Linear(in_features=512, out_features=2048, bias=True)
        (proj): Linear(in_features=2048, out_features=512, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=50257, bias=False)
)

In [38]:
model.eval()

GPT(
  (wte): Embedding(50257, 512)
  (wpe): Embedding(128, 512)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-7): 8 x Block(
      (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=512, out_features=1536, bias=True)
        (c_proj): Linear(in_features=512, out_features=512, bias=True)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc): Linear(in_features=512, out_features=2048, bias=True)
        (proj): Linear(in_features=2048, out_features=512, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=50257, bias=False)
)

In [39]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [40]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<|endoftext|>
50256


In [41]:
pad_id = tokenizer.eos_token_id

In [42]:
prompts = [
    "She loved to run and play and have fun. One day, she was running and she fell down. It",
    "playing in the rain. But then she saw something else in the rain. She panicked. Her mommy tried to help h",
    "it and she went inside the house. Lily looked",
    "The bird was flying very fast and Lily's mommy told her it was time to go outside.",
    "It was a big, mean cat and it made her cry. She was scared and scared.",
    "The cat just shrugged and walked away into the woods. She was scared",
    "she would go back and hide in the forest. But then she heard a voice.",
    "It was her neighbor! He was a nice man and he said,",
    "Don't worry, I'm not scared. I will get you out soon.",
    "The little girl opened her eyes and saw his neighbor, Mr. Jones and his",
    "Once upon a time, "
]

tokens = [tokenizer.encode(p) for p in prompts]

max_len = max(len(t) for t in tokens)

padded = [
    t + [pad_id] * (max_len - len(t))
    for t in tokens
]

idx = torch.tensor(
    padded + padded + padded,
    dtype=torch.long,
    device=device
)

In [43]:
idx.shape

torch.Size([33, 25])

In [44]:
import time
start = time.perf_counter()

out = model.generate(
    idx,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50
)

end = time.perf_counter()

print(f"Generation time: {end - start:.3f} seconds")

Generation time: 7.571 seconds


In [ ]:
text = tokenizer.decode(
    out[5].tolist()
)

print(text)

The cat just shrugged and walked away into the woods. She was scared<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>Once there was a little girl named Lily who loved to play with her toys. One day, she found a big round ball in her backyard. It was so tall that it made her smile. 

Lily showed the ball to her friend, a boy named Timmy. "Look, Timmy!" said Lily.

Timmy looked at the ball and said, "Wow! It's so pretty."

Lily smiled and said, "I think it's pretty!


## **Model with KV cache**

In [46]:
# gpt2_min.py
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import pdb

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1

class CausalSelfAttentionKV(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.resid_drop = nn.Dropout(config.dropout)
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self, x, past_k=None, past_v=None):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        head_dim = C // self.n_head

        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        if past_k is not None:
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        total_len = k.size(2)

        att = (q @ k.transpose(-2, -1)) * (
            1.0 / math.sqrt(head_dim)
        )

        if T > 1:
            att = att.masked_fill(
                self.bias[:, :, :T, :total_len] == 0,
                float("-inf")
            )

        att = F.softmax(att, dim=-1)

        y = att @ v

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(B, T, C)
        )

        y = self.resid_drop(self.c_proj(y))

        return y, k, v

class MLPKV(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class BlockKV(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttentionKV(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLPKV(config)
    def forward(self, x, past_k=None, past_v=None):

        attn_out, k, v = self.attn(
            self.ln_1(x),
            past_k,
            past_v
        )

        x = x + attn_out

        x = x + self.mlp(self.ln_2(x))

        return x, k, v

class GPTKV(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([BlockKV(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None, past_kvs=None):
        if past_kvs is None:
            past_kvs = [None] * len(self.h)

        new_kvs = []
        B, T = idx.size()
        # pdb.set_trace()
        past_len = 0

        if past_kvs[0] is not None:
            past_len = past_kvs[0][0].size(2)

        pos = torch.arange(
            past_len,
            past_len + T,
            device=idx.device
        ).unsqueeze(0)
        x = self.wte(idx) + self.wpe(pos)
        x = self.drop(x)
        for block, past in zip(self.h, past_kvs):

            if past is None:
                pk, pv = None, None
            else:
                pk, pv = past

            x, k, v = block(
                x,
                pk,
                pv
            )

            new_kvs.append((k, v))
        x = self.ln_f(x)
        logits = self.lm_head(x)
        # pdb.set_trace()
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        # pdb.set_trace()
        return logits, loss, new_kvs


    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens=50,
        temperature=1.0,
        top_k=None
    ):

        past_kvs = None

        logits, _, past_kvs = self(
            idx,
            past_kvs=past_kvs
        )

        for _ in range(max_new_tokens):

            logits = logits[:, -1, :]
            logits = logits / max(
                temperature,
                1e-8
            )

            if top_k is not None:
                v, _ = torch.topk(
                    logits,
                    min(top_k, logits.size(-1))
                )

                thresh = v[:, -1].unsqueeze(-1)

                logits = torch.where(
                    logits < thresh,
                    torch.full_like(
                        logits,
                        -float("inf")
                    ),
                    logits
                )

            probs = F.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                [idx, next_token],
                dim=1
            )

            logits, _, past_kvs = self(
                next_token,
                past_kvs=past_kvs
            )

        cache_bytes = 0

        for k, v in past_kvs:
            cache_bytes += k.numel() * k.element_size()
            cache_bytes += v.numel() * v.element_size()

        print(
            f"KV Cache Size: "
            f"{cache_bytes / (1024**2):.2f} MB"
        )

        for i, (k, v) in enumerate(past_kvs):
            layer_bytes = (
                k.numel() * k.element_size()
                + v.numel() * v.element_size()
            )

            print(
                f"Layer {i}: "
                f"{layer_bytes / (1024**2):.2f} MB"
            )

        return idx

In [47]:
modelKV = GPTKV(config)
modelKV.load_state_dict(state_dict)
modelKV.to(device)
modelKV.eval()

GPTKV(
  (wte): Embedding(50257, 512)
  (wpe): Embedding(128, 512)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-7): 8 x BlockKV(
      (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttentionKV(
        (c_attn): Linear(in_features=512, out_features=1536, bias=True)
        (c_proj): Linear(in_features=512, out_features=512, bias=True)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLPKV(
        (fc): Linear(in_features=512, out_features=2048, bias=True)
        (proj): Linear(in_features=2048, out_features=512, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=50257, bias=False)
)

In [ ]:
idx.shape

torch.Size([33, 25])

# **Without KV cache**

In [53]:
import time
start = time.perf_counter()
out = model.generate(idx, max_new_tokens=100, temperature=0.8, top_k=50)
end = time.perf_counter()
print(f"Generation time: {end - start:.3f} seconds")

Generation time: 8.225 seconds


# **With KV cache**

In [51]:
import time
start = time.perf_counter()

out = modelKV.generate(idx, max_new_tokens=100, temperature=0.8, top_k=50)
end = time.perf_counter()
print(f"Generation time with KV cache: {end - start:.3f} seconds")

KV Cache Size: 128.91 MB
Layer 0: 16.11 MB
Layer 1: 16.11 MB
Layer 2: 16.11 MB
Layer 3: 16.11 MB
Layer 4: 16.11 MB
Layer 5: 16.11 MB
Layer 6: 16.11 MB
Layer 7: 16.11 MB
Generation time with KV cache: 0.755 seconds
